# 07. 근거 검색(B안) + 로컬 LLM 진단(C안) (2026-11-19)

- D53(B안 시연), D56(사람 확인 → C안), D60(원문 외부 전송 금지, 로컬 LLM).
- 모델: 진단 exaone3.5:7.8b, 임베딩 bge-m3 (Ollama, 이 컴퓨터 안에서만). 벡터 저장소 FAISS (네트워크 전송 없음).
- 규칙 판정(03)이 먼저이고, LLM은 검토만 한다. 규칙과 다르거나 확신도 낮음·근거 부족·형식 실패·인용 오류는 사람 확인.
- 판정 없이 사실만 출력한다. 원문 문장·LLM 근거 요약은 `data/llm_cache/`(git 제외)에만 둔다.

데이터: AI Hub(한국지능정보사회진흥원) 사업 결과

In [ ]:
import sys, json, time
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
from src.rag import assert_local, ATTRIBUTION
from src.rag import regulations, index as rindex
from src.rag.cases import build_cases, product_cases
from src.rag.diagnose import diagnose, needs_human, _llm
from src.rag.evidence import write_card
from src.load_cot import load_cot, load_products
from src.judge import judge
assert_local()
AS_OF, COMPANY = "2026-11-19", "한국투자증권"
RERUN = False   # True면 캐시를 무시하고 LLM을 다시 돌린다
CACHE_DIR = ROOT / "data" / "llm_cache"; OUT_DIR = ROOT / "outputs" / "llm"
OUT_DIR.mkdir(parents=True, exist_ok=True)
cot = load_cot(); prod = load_products()
j, ctx = judge(cot, prod, as_of="2026-11-18")
print("판정:", len(j))

## 1. 규정 문서 → 조항 단위 → FAISS 색인

In [ ]:
reg_index = regulations.load_index()
chunks = regulations.load_chunks(reg_index)
print(chunks.groupby(["층위", "doc_id"]).size().to_string())
t0 = time.time()
store = rindex.build(chunks)
print(f"색인: {len(chunks)} 조항, {time.time() - t0:.0f}초")

## 2. 검증 사례 (사람 판정이 있는 표본)

| 표본 | 사례 유형 | 사람 판정 |
|---|---|---|
| 범위밖_분류표 32 | 범위 | 처리(제외 여부), 내 분류(사유 코드) |
| 희망검수표 1·2 40 | 이관 | 내 판정(분류). 희망검수표2는 20건 모두 결함, 희망검수표1(회피·오탐)은 결함 아님 |
| 이관검수표 20 | 이관 | 내 분류, 내 판정 |
| 원천검수표 20 | 원천(상품 단위) | 오류 4 / 오탐 16 |

표본별 한계: 범위밖 32는 D57을, 희망 40은 이관 키워드를, 이관 20은 경험 규칙을 정할 때 본 표본이다 (규칙 일치율이 낙관적). 원천 20은 H1 키워드 후보에서 뽑았다.
프롬프트에는 이 표본에서 뽑은 예시를 넣지 않는다.

In [ ]:
rd = lambda p: pd.read_csv(ROOT / "data" / "review" / p, encoding="utf-8-sig", dtype=str, keep_default_na=False)
rv, h1, h2, t7, sv = rd("범위밖_분류표.csv"), rd("희망검수표.csv"), rd("희망검수표2.csv"), rd("이관검수표.csv"), rd("원천검수표.csv")
MAP = {"희망(명확)": "R1 명확한 희망", "고려 중": "R2 고려 중", "경험": "경험", "비교 표현": "R4 비교 표현",
       "회피": "회피", "오탐(일반 표현)": "일반 표현"}
human = []
for _, x in rv.iterrows():
    human.append({"표본": "범위밖32", "case_id": f"{x['cot_id']}:범위", "사람 결함": x["처리"] != "교정 + 태그",
                  "사람 처리": "제외" if x["처리"] == "제외" or x["처리"].startswith("엄격본 제외") else x["처리"], "사람 코드": x["내 분류"]})
for df, name, defect in [(h1, "희망1", lambda x: False), (h2, "희망2", lambda x: True), (t7, "이관20", lambda x: x["내 판정"] == "결함")]:
    for _, x in df.iterrows():
        human.append({"표본": name, "case_id": f"{x['cot_id']}:이관", "사람 결함": defect(x), "사람 분류": MAP.get(x["내 판정"] if name.startswith("희망") else x["내 분류"], "")})
for i, x in sv.reset_index(drop=True).iterrows():
    human.append({"표본": "원천20", "case_id": f"P{i:02d}:원천", "사람 결함": x["내 판정"] != "일치(오탐)"})
human = pd.DataFrame(human)

ids = {int(c.split(":")[0]) for c in human["case_id"] if not c.startswith("P")}
cases = [c for c in build_cases(j, ctx, prod, ids=ids) if c["case_id"] in set(human["case_id"])]
# 규칙상 그 유형이 아닌 CoT도 사람 판정이 있으면 그 유형으로 만든다
have = {c["case_id"] for c in cases}
for cid_t in sorted(set(human["case_id"]) - have):
    if cid_t.startswith("P"):
        continue
    cid, t = cid_t.split(":")
    cases += build_cases(j, ctx, prod, ids=[int(cid)], force_types=[t])
cases += product_cases(sv["상품명"].tolist(), prod)
print("검증 사례:", len(cases), pd.Series([c["유형"] for c in cases]).value_counts().to_dict(),
      "/ 사람 판정 행:", len(human), "/ 사례 없음:", sorted(set(human["case_id"]) - {c["case_id"] for c in cases}))

## 3. 근거 검색 + LLM 진단 (검증 사례)

- 기준일 `AS_OF`, 회사 기준 `COMPANY`로 시행 중인 조항만 검색해 상위 5개를 근거로 넣는다.
- 결과는 `data/llm_cache/verify_results.csv`에 캐시한다 (RERUN=False면 다시 돌리지 않음).

In [ ]:
cache = CACHE_DIR / "verify_results.csv"
if cache.exists() and not RERUN:
    res = pd.read_csv(cache, encoding="utf-8-sig", keep_default_na=False)
    print("캐시 사용:", len(res))
else:
    llm = _llm()
    rows, t0 = [], time.time()
    for i, c in enumerate(cases):
        ev = rindex.search(store, c["질의"], AS_OF, COMPANY, k=5)
        r = diagnose(c, ev, llm)
        r["검색 top3"] = "|".join(e[0] for e in ev[:3])
        rows.append(r)
        if (i + 1) % 20 == 0:
            print(i + 1, "/", len(cases), f"{time.time() - t0:.0f}초")
    res = pd.DataFrame(rows)
    res.to_csv(cache, index=False, encoding="utf-8-sig")
print("상태:", res["상태"].value_counts().to_dict(), "/ 재시도 있음:", (res["재시도"] > 0).sum(),
      f"/ 평균 {res['소요(초)'].mean():.1f}초, 최대 {res['소요(초)'].max():.1f}초")

## 4. 검증 지표

In [ ]:
m = human.merge(res, on="case_id", how="left")
ok = m["상태"] == "정상"
m["LLM 결함"] = m["결함"] == "결함"
m["결함 일치"] = ok & (m["LLM 결함"] == m["사람 결함"])
m["분류 일치"] = ok & (m["이관 분류"] == m["사람 분류"])
m["처리 일치"] = ok & ((m["처리"] == "제외") == (m["사람 처리"] == "제외"))
m["코드 일치"] = ok & (m["사유 코드"] == m["사람 코드"])
rows = []
for name, g in m.groupby("표본", sort=False):
    rows.append({"표본": name, "크기": len(g), "형식 실패": (g["상태"] != "정상").sum(),
                 "결함 일치": g["결함 일치"].sum(),
                 "분류 일치": g["분류 일치"].sum() if g["사람 분류"].fillna("").ne("").any() else "-",
                 "처리 일치": g["처리 일치"].sum() if name == "범위밖32" else "-",
                 "코드 일치": g["코드 일치"].sum() if name == "범위밖32" else "-",
                 "인용 오류": g["인용 오류"].fillna(False).astype(bool).sum(),
                 "근거 부족": (g["결함"] == "근거 부족").sum()})
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print()
print("[확신도별 결함 일치]")
print(m[ok].groupby("확신도")["결함 일치"].agg(["size", "sum", "mean"]).round(2).to_string())

In [ ]:
# 사람이 봐야 할 사례 수: 규칙만 vs 규칙 + LLM
rule_human = {"범위": lambda r: str(r["기준③"]).startswith("판정 불가"), "이관": lambda r: r["이관 판정"] == "사람 확인", "원천": lambda r: False}
by_case = {c["case_id"]: c for c in cases}
def rule_defect(c):
    r = c["규칙"]
    if c["유형"] == "범위": return r.get("처리") == "제외"
    if c["유형"] == "이관": return r.get("이관 판정") == "결함"
    return r.get("원천 확인 필요(body_grade)") if r.get("원천 확인 필요(body_grade)") is not None else True
res["사람 확인 이유"] = [needs_human(r, rule_defect(by_case[r["case_id"]])) for r in res.to_dict("records")]
n_rule = sum(rule_human[c["유형"]](c["규칙"]) for c in cases if c["유형"] != "원천")
print("규칙만으로 사람 확인:", n_rule, "/", len(cases))
print("규칙 + LLM으로 사람 확인:", (res["사람 확인 이유"] != "").sum(), "/", len(res))
print(res["사람 확인 이유"].replace("", "(없음)").value_counts().to_string())

In [ ]:
# 검색: 정답 조항이 상위 3개 안에 있는 비율 (gold_articles.csv에 조항이 적힌 유형만)
gold = pd.read_csv(ROOT / "src" / "rag" / "gold_articles.csv", encoding="utf-8", dtype=str, keep_default_na=False)
gold = gold[(gold["조항"] != "") & gold["doc_id"].isin(set(chunks["doc_id"]))]
def hit(r):
    g = gold[gold["유형"] == r["유형"]]
    if g.empty: return None
    top = str(r["검색 top3"]).split("|")
    return any(t.startswith(f"{d}:{a}") for d, a in zip(g["doc_id"], g["조항"]) for t in top)
res["정답 조항 top3"] = res.apply(hit, axis=1)
print("정답 조항 정의가 있는 사례:", res["정답 조항 top3"].notna().sum())
print(res.dropna(subset=["정답 조항 top3"]).groupby("유형")["정답 조항 top3"].agg(["size", "sum"]).to_string())
cited = res[res["상태"] == "정상"]
print("인용 조항이 검색 결과 안에 있는 비율:", f"{(~cited['인용 오류'].astype(bool)).mean() * 100:.1f}%")
print("JSON 형식: 첫 시도 실패", (res["재시도"] > 0).sum(), "/ 재시도 후 실패", (res["상태"] != "정상").sum(), "/", len(res))

In [ ]:
# 결과 저장 (원문·근거 요약 제외)
keep = ["case_id", "cot_id", "유형", "분할", "상태", "재시도", "결함", "처리", "사유 코드", "이관 분류", "확신도",
        "인용 조항", "인용 오류", "검색 top3", "정답 조항 top3", "사람 확인 이유", "소요(초)"]
res[[k for k in keep if k in res.columns]].to_csv(OUT_DIR / "diagnoses_verify.csv", index=False, encoding="utf-8-sig")
summary.to_csv(OUT_DIR / "verify_summary.csv", index=False, encoding="utf-8-sig")
# 근거 카드: 사람 확인으로 간 사례 중 5건만 예시로
for c in [by_case[i] for i in res.loc[res["사람 확인 이유"] != "", "case_id"].head(5)]:
    ev = rindex.search(store, c["질의"], AS_OF, COMPANY, k=5)
    print(write_card(c, res[res["case_id"] == c["case_id"]].iloc[0].to_dict(), ev, AS_OF, COMPANY))

## 5. 기준일·회사 기준을 바꾸면 근거와 판정이 바뀌는 시연

- 같은 사례 5건을 (공개기준·한국투자증권, as_of 오늘) / (KB국민은행 기준) 으로 검색·판정해 나란히 본다.
- 판정 함수 쪽은 `d43_range.csv`의 기준명(공개기준 / KB국민은행)을 바꿔 `judge(…, standard=…)`로 다시 돌린다.

In [ ]:
demo_ids = [c["case_id"] for c in cases if c["유형"] == "범위"][:5]
rows = []
for cid in demo_ids:
    c = by_case[cid]
    for comp in ["한국투자증권", "KB국민은행"]:
        ev = rindex.search(store, c["질의"], AS_OF, comp, k=5)
        rows.append({"case_id": cid, "회사 기준": comp, "검색 top3": "|".join(e[0] for e in ev[:3])})
print(pd.DataFrame(rows).to_string(index=False))

---
데이터: AI Hub(한국지능정보사회진흥원) 사업 결과